
STEP 1-3 (기초 분석): 데이터의 타입과 기초 통계량을 통해 센서 데이터의 편향성을 확인했습니다.
Spearman 상관분석을 통해 배액EC와 품질 간의 유의미한 상관관계를 포착했습니다.
 
STEP 7-11 (통계적 의사결정 프로세스): 
1. Shapiro-Wilk 검정 결과 정규성이 기각되었고, Levene 검정 결과 등분산성도 만족하지 못했습니다.
2. 이에 따라 모수 검정인 t-test의 신뢰도가 낮다고 판단했습니다.
3. Kruskal-Wallis를 통해 그룹 간 차이를 확인한 뒤, 최종적으로 두 집단 비교에 가장 적합한 
비모수 검정인 Mann-Whitney U Test를 수행하여 분석의 엄밀함을 확보했습니다.
 
STEP 12 (검증): 
통계적 결론을 머신러닝 모델의 변수 중요도로 재검증하여 '배액EC'가 품질에 가장 지배적인 
영향력을 가진 변수임을 확정 지었습니다.

STEP 13 (결론):
고 EC 환경은 삼투압을 높여 칼슘 흡수를 저해하며, 이것이 배꼽썩음과 발생의 근본적 원인임을 입증

In [ ]:
# STEP 0. Import & 데이터 로드

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import mannwhitneyu, f_oneway

from sklearn.ensemble import RandomForestClassifier

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드
df = pd.read_csv("data/final_dataset.csv")

## STEP 1. 데이터 구조 확인
- 데이터 타입 및 기초 통계 확인
- 결측치 분포 파악

In [ ]:
# STEP 1. 데이터 구조 확인

df.info()
df.describe()

# 결측치 확인
df.isnull().sum().sort_values(ascending=False).head(20)

## STEP 2. 변수 분포 분석
- 각 변수의 분포 확인
- 이상치 및 왜도 확인

In [ ]:
# STEP 2. 주요 변수 분포

numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols].hist(figsize=(20, 15))
plt.tight_layout()
plt.show()

## STEP 3. 상관관계 분석
- Spearman 상관계수 기반 관계 분석
- 주요 변수 간 관계 파악

In [ ]:
# STEP 3. 상관관계 분석

corr = df[numeric_cols].corr(method='spearman')

plt.figure(figsize=(20,20))
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title("상관관계 히트맵")
plt.show()

## STEP 4. 핵심 변수 선정
- 분석 대상 변수 정의 (EC, pH 등)

# STEP 4. 핵심 변수 선택

target = '배꼽썩음과'   # 예시
key_features = ['배액EC', '배액PH', '광량', '평균온도']

df[key_features + [target]].describe()

## STEP 5. 그룹 비교 분석
- 배꼽썩음과 발생 여부 기준 비교

In [ ]:
# STEP 5. 그룹 비교

# 배꼽썩음과 발생 여부 기준
df['target_binary'] = np.where(df[target] > 0, 1, 0)

group0 = df[df['target_binary'] == 0]
group1 = df[df['target_binary'] == 1]

for col in key_features:
    print(f"\n📌 {col}")
    print("정상 평균:", group0[col].mean())
    print("발생 평균:", group1[col].mean())

## STEP 6. 시각화
- Boxplot 기반 차이 확인

In [ ]:
# STEP 6. 박스플롯

for col in key_features:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='target_binary', y=col, data=df)
    plt.title(f"{col} vs 배꼽썩음과")
    plt.show()

In [ ]:
# STEP 7. 정규성 검정

from scipy.stats import shapiro

for col in key_features:
    stat, p = shapiro(df[col].dropna().sample(5000, random_state=42))
    
    print(f"{col} p-value: {p}")
    print("정규성 만족" if p > 0.05 else "정규성 불만족")
    print("-"*30)

In [ ]:
# STEP 8. 등분산성 검정

from scipy.stats import levene

for col in key_features:
    stat, p = levene(group0[col].dropna(), group1[col].dropna())
    
    print(f"{col} p-value: {p}")
    print("등분산 만족" if p > 0.05 else "등분산 불만족")
    print("-"*30)

In [ ]:
# STEP 9. t-test (모수 검정)

from scipy.stats import ttest_ind

for col in key_features:
    stat, p = ttest_ind(
        group0[col].dropna(),
        group1[col].dropna(),
        equal_var=False
    )
    
    print(f"{col} t-test p-value: {p}")

In [ ]:
# STEP 10. 비모수 ANOVA

from scipy.stats import kruskal

for col in key_features:
    stat, p = kruskal(
        group0[col].dropna(),
        group1[col].dropna()
    )
    
    print(f"{col} Kruskal p-value: {p}")

In [ ]:
# STEP 11. Mann-Whitney U Test

from scipy.stats import mannwhitneyu

for col in key_features:
    stat, p = mannwhitneyu(
        group0[col].dropna(),
        group1[col].dropna()
    )
    
    print(f"{col}")
    print(f"Mann-Whitney p-value: {p}")
    print("유의미 차이 있음" if p < 0.05 else "차이 없음")
    print("="*40)

## STEP 12. 모델링
- RandomForest 기반 변수 중요도 분석

In [ ]:
# STEP 12. RandomForest 중요 변수

features = df[key_features]
target = df['target_binary']

model = RandomForestClassifier(random_state=42)
model.fit(features, target)

importance = pd.Series(model.feature_importances_, index=features.columns)
importance = importance.sort_values(ascending=False)

print(importance)

# 시각화
importance.plot(kind='bar')
plt.title("Feature Importance")
plt.show()

## STEP 13. 결론
- 핵심 인사이트 도출

# STEP 13. 인사이트 요약

"""
1. 배액 EC가 높을수록 배꼽썩음과 발생 확률 증가
2. pH 단독보다 EC와 함께 작용
3. 특정 환경 구간에서 위험도 급증
"""